<a href="https://colab.research.google.com/github/juliocontreras-alt/Colab/blob/main/Actualizar_ComprasMP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# COLAB: ENVIAR COMPRAS DEL CSV A ARCHIVOS DE GOOGLE SHEETS
# ============================================================

!pip -q install gspread google-auth google-auth-oauthlib google-auth-httplib2 pandas openpyxl

import pandas as pd
import re
import unicodedata
from google.colab import files, auth
from google.auth import default
import gspread
from googleapiclient.discovery import build


In [ ]:
# 1) AUTENTICACIÓN
# =========================================================
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
drive_service = build("drive", "v3", credentials=creds)

# =========================================================

In [ ]:
# 2) CONFIGURACIÓN
# =========================================================
ROOT_FOLDER_ID = "1jHwWEQjM7Fb1mt_H5YlmelNO6dbm45mx"
TARGET_YEAR = "2026"
TARGET_SHEET_NAME = "REMISIONES"
START_ROW = 5

# Mapeo Deliver To -> carpeta en Drive
# La llave debe ser el Deliver To normalizado
plant_mapping = {
    "NEO STRECH SA DE CV MTY": {
        "folder_name": "NEOSTRETCH MTY",
        "file_hint": "MTY"
    },
    "AREXA ARPILLAS SA DE CV": {
        "folder_name": "AREXA ARPILLAS",
        "file_hint": "ARPILLAS"
    },
    "ARPILLAS DE EXPORTACION SA DE CV": {
        "folder_name": "ARPILLAS DE EXPORTACIÓN",
        "file_hint": "ARPILLAS"
    },
    "CARATREN": {
        "folder_name": "CARATREN",
        "file_hint": "CARATREN"
    },
    "INTERNACIONAL DE SACOS Y ARPILLAS SA DE CV": {
        "folder_name": "ISA",
        "file_hint": "ISA"
    },
    "NEOPACK": {
        "folder_name": "NEOPACK",
        "file_hint": "NEOPACK"
    },
    "NEO PELICULAS SA DE CV": {
        "folder_name": "NEOPELICULAS COMPRAS",
        "file_hint": "STRETCH"
    },
    "NEO STRECH SA DE CV CORREO": {
        "folder_name": "NEOSTRETCH CORREO",
        "file_hint": "CORREO"
    },

    "NEOSTRETCH MTY ESQ": {
        "folder_name": "NEOSTRETCH MTY ESQ",
        "file_hint": "MTY"
    },
    "SAIISA": {
        "folder_name": "SAIISA",
        "file_hint": "SAIISA"
    },
    "MAT SACOS DE POLIPROPILENO ESPECIALIZADOS SA DE CV": {
        "folder_name": "SAPIESA MATRIZ",
        "file_hint": "SAPIESA"
    },
    "SUC SACOS DE POLIPROPILENO ESPECIALIZADOS SA DE CV": {
        "folder_name": "SAPIESA SUCURSAL",
        "file_hint": "SAPIESA"
    }
}

# Si quieres limpiar la hoja antes de pegar
CLEAR_BEFORE_WRITE = True


In [ ]:
# 3) FUNCIONES AUXILIARES
# =========================================================
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip().upper()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_money(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().replace("$", "").replace(",", "")
    if value == "":
        return ""
    try:
        return float(value)
    except:
        return value

def find_folder_by_name(folder_name, parent_id):
    query = (
        f"'{parent_id}' in parents and "
        f"mimeType='application/vnd.google-apps.folder' and "
        f"name='{folder_name}' and trashed=false"
    )
    result = drive_service.files().list(
        q=query,
        fields="files(id, name)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ).execute()
    files_found = result.get("files", [])
    return files_found[0]["id"] if files_found else None

def find_month_file(parent_id, month_prefix, file_hint=None):
    query = (
        f"'{parent_id}' in parents and "
        f"mimeType='application/vnd.google-apps.spreadsheet' and "
        f"trashed=false"
    )
    result = drive_service.files().list(
        q=query,
        fields="files(id, name)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ).execute()

    files_found = result.get("files", [])
    candidates = [f for f in files_found if f["name"].strip().startswith(f"{month_prefix}.")]

    if file_hint:
        hint_norm = normalize_text(file_hint)
        candidates_hint = [f for f in candidates if hint_norm in normalize_text(f["name"])]
        if candidates_hint:
            return candidates_hint[0]

    return candidates[0] if candidates else None

def get_next_empty_row(ws, start_row=5):
    values = ws.get(f"A{start_row}:A")
    return start_row + len(values)

In [ ]:
# 4) SUBIR CSV
# =========================================================
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]

df = pd.read_csv(csv_filename, sep=None, engine="python")

print("Columnas detectadas:")
print(df.columns.tolist())

# =========================================================

Saving 18032026.csv to 18032026.csv
Columnas detectadas:
['INVOICE', 'Ship Date', 'Deliver To', 'REM', 'PRODUCTO', 'PRECIO / LB', 'TOTAL FACTURA', 'GASTOS ADUANALES']


In [ ]:
# 5) VALIDAR COLUMNAS
# =========================================================
required_cols = [
    "INVOICE",
    "Ship Date",
    "Deliver To",
    "REM",
    "PRODUCTO",
    "PRECIO / LB",
    "TOTAL FACTURA"
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas en el CSV: {missing}")

# =========================================================-

In [ ]:
# 6) PREPARAR DATOS
# =========================================================
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True, errors="coerce")
df = df[df["Ship Date"].notna()].copy()

df["DELIVER_TO_NORM"] = df["Deliver To"].apply(normalize_text)
df["MES_NUM"] = df["Ship Date"].dt.month.astype(int).astype(str).str.zfill(2)

# =========================================================
# ------------------------------------------------------------

In [ ]:
# 7) AGRUPAR POR PLANTA + MES
# =========================================================
grouped = df.groupby(["DELIVER_TO_NORM", "MES_NUM"])

log_ok = []
log_error = []
not_mapped = set()

for (deliver_norm, mes_num), subdf in grouped:
    if deliver_norm not in plant_mapping:
        not_mapped.add(deliver_norm)
        continue

    mapping = plant_mapping[deliver_norm]
    folder_name = mapping["folder_name"]
    file_hint = mapping.get("file_hint", "")

    try:
        # Buscar carpeta planta
        plant_folder_id = find_folder_by_name(folder_name, ROOT_FOLDER_ID)
        if not plant_folder_id:
            log_error.append(f"No se encontró la carpeta de planta: {folder_name}")
            continue

        # Buscar carpeta 2026
        year_folder_id = find_folder_by_name(TARGET_YEAR, plant_folder_id)
        if not year_folder_id:
            log_error.append(f"No se encontró la carpeta {TARGET_YEAR} dentro de {folder_name}")
            continue

        # Buscar archivo del mes
        month_file = find_month_file(year_folder_id, mes_num, file_hint)
        if not month_file:
            log_error.append(f"No se encontró archivo mes {mes_num} en {folder_name}")
            continue

        spreadsheet_id = month_file["id"]
        spreadsheet_name = month_file["name"]

        sh = gc.open_by_key(spreadsheet_id)
        ws = sh.worksheet(TARGET_SHEET_NAME)

        # Preparar datos A:M
        rows = []
        for _, row in subdf.iterrows():
            out = [""] * 13  # A:M

            out[0] = "" if pd.isna(row["INVOICE"]) else str(row["INVOICE"])  # A
            out[1] = row["Ship Date"].strftime("%d/%m/%Y") if pd.notna(row["Ship Date"]) else ""  # B
            out[2] = "" if pd.isna(row["Deliver To"]) else str(row["Deliver To"])  # C
            out[3] = "" if pd.isna(row["REM"]) else str(row["REM"])  # D
            out[4] = "" if pd.isna(row["PRODUCTO"]) else str(row["PRODUCTO"])  # E
            # F:K vacías
            out[11] = clean_money(row["PRECIO / LB"])  # L
            out[12] = clean_money(row["TOTAL FACTURA"])  # M

            rows.append(out)

        if CLEAR_BEFORE_WRITE:
            ws.batch_clear([f"A{START_ROW}:M1000"])
            target_row = START_ROW
        else:
            target_row = get_next_empty_row(ws, START_ROW)

        end_row = target_row + len(rows) - 1
        write_range = f"A{target_row}:M{end_row}"

        ws.update(write_range, rows, value_input_option="USER_ENTERED")

        log_ok.append(
            f"OK | Planta CSV: {deliver_norm} | Archivo: {spreadsheet_name} | "
            f"Mes: {mes_num} | Filas: {len(rows)} | Rango: {write_range}"
        )

    except Exception as e:
        log_error.append(
            f"ERROR | Planta CSV: {deliver_norm} | Mes: {mes_num} | Detalle: {str(e)}"
        )

# =========================================================

/tmp/ipykernel_207/2347850681.py:68: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws.update(write_range, rows, value_input_option="USER_ENTERED")
/tmp/ipykernel_207/2347850681.py:68: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws.update(write_range, rows, value_input_option="USER_ENTERED")
/tmp/ipykernel_207/2347850681.py:68: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  ws.update(write_range, rows, value_input_option="USER_ENTERED")
/tmp/ipykernel_207/2347850681.py:68: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arg

In [ ]:
# 8) RESULTADOS
# =========================================================
print("\n" + "="*80)
print("RESULTADOS")
print("="*80)

if log_ok:
    print("\nPROCESADOS CORRECTAMENTE:")
    for item in log_ok:
        print(item)

if not_mapped:
    print("\nSIN MAPEO EN plant_mapping:")
    for item in sorted(not_mapped):
        print("-", item)

if log_error:
    print("\nERRORES:")
    for item in log_error:
        print(item)


RESULTADOS

PROCESADOS CORRECTAMENTE:
OK | Planta CSV: AREXA ARPILLAS SA DE CV | Archivo: 03. FORMATO COMPRAS CIERRE 2026 - ARPILLAS | Mes: 03 | Filas: 2 | Rango: A5:M6
OK | Planta CSV: ARPILLAS DE EXPORTACION SA DE CV | Archivo: 03. FORMATO COMPRAS CIERRE 2026 - EXPORTACIÓN | Mes: 03 | Filas: 3 | Rango: A5:M7
OK | Planta CSV: INTERNACIONAL DE SACOS Y ARPILLAS SA DE CV | Archivo: 03. FORMATO COMPRAS CIERRE 2026 - ISA | Mes: 03 | Filas: 4 | Rango: A5:M8
OK | Planta CSV: MAT SACOS DE POLIPROPILENO ESPECIALIZADOS SA DE CV | Archivo: 03. FORMATO COMPRAS CIERRE 2026 - SAPM | Mes: 03 | Filas: 6 | Rango: A5:M10
OK | Planta CSV: NEO PELICULAS SA DE CV | Archivo: 03. FORMATO COMPRAS CIERRE 2026 - STRETCH | Mes: 03 | Filas: 14 | Rango: A5:M18
OK | Planta CSV: NEO STRECH SA DE CV CORREO | Archivo: 03. FORMATO COMPRAS CIERRE 2026 - CORREO | Mes: 03 | Filas: 7 | Rango: A5:M11
OK | Planta CSV: NEO STRECH SA DE CV MTY | Archivo: 03. FORMATO COMPRAS CIERRE 2026 - MTY | Mes: 03 | Filas: 19 | Rango: A5